<a href="https://colab.research.google.com/github/vipinsinghbora03-dot/Python_Project_28/blob/main/project_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

# Creating synthetic/mock data because '28_tournament_match_stats.csv' is missing
try:
    df = pd.read_csv("28_tournament_match_stats.csv")
except FileNotFoundError:
    print("Warning: CSV file not found. Generating a mock dataset for demonstration.\n")
    np.random.seed(42)
    n_rows = 100

    # Generate dates
    dates = pd.date_range(start="2023-01-01", periods=30, freq="D")

    df = pd.DataFrame({
        'Match_ID': [f"M_{1000 + i}" for i in range(n_rows)],
        'Match_Date': np.random.choice(dates, size=n_rows),
        'Player': np.random.choice(["Aarav Sharma", "Vihaan Patel", "Kabir Singh", "Diya Sen", "Ananya Roy"], size=n_rows),
        'Team': np.random.choice(["Delhi Strikers", "Mumbai Titans", "Kolkata Knights", "Bangalore Royals"], size=n_rows),
        'Points': np.random.randint(50, 200, size=n_rows).astype(float),
        'Match_Duration': np.random.randint(60, 180, size=n_rows), # stored as minutes
        'Result': np.random.choice(["Win", "Loss", "Draw"], size=n_rows, p=[0.45, 0.45, 0.10])
    })

    # Inject some missing values to demonstrate cleaning
    df.loc[np.random.choice(n_rows, 5, replace=False), 'Points'] = np.nan
    # Map schema aliases to allow existing downstream code to work
    df['Player_Points'] = df['Points']
    df['Match_Duration_Mins'] = df['Match_Duration']

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nData Types:")
print(df.dtypes)
print("\nFirst 5 Rows:")
print(df.head())


Dataset Shape: (100, 9)

Columns:
Index(['Match_ID', 'Match_Date', 'Player', 'Team', 'Points', 'Match_Duration',
       'Result', 'Player_Points', 'Match_Duration_Mins'],
      dtype='object')

Data Types:
Match_ID                       object
Match_Date             datetime64[ns]
Player                         object
Team                           object
Points                        float64
Match_Duration                  int64
Result                         object
Player_Points                 float64
Match_Duration_Mins             int64
dtype: object

First 5 Rows:
  Match_ID Match_Date        Player              Team  Points  Match_Duration  \
0   M_1000 2023-01-07  Vihaan Patel  Bangalore Royals    91.0             109   
1   M_1001 2023-01-20      Diya Sen   Kolkata Knights   148.0              84   
2   M_1002 2023-01-29      Diya Sen   Kolkata Knights    56.0              83   
3   M_1003 2023-01-15  Aarav Sharma     Mumbai Titans   193.0              72   
4   M_1004 2023-0

In [6]:
#  Check for missing values and duplicate Match_ID records

# Check missing values
print(df.isnull().sum())

# Check duplicate Match_ID
print(df["Match_ID"].duplicated().sum())

Match_ID               0
Match_Date             0
Player                 0
Team                   0
Points                 5
Match_Duration         0
Result                 0
Player_Points          5
Match_Duration_Mins    0
dtype: int64
0


In [7]:
# 1. Remove duplicate records based on Match_ID
if 'Match_ID' in df.columns:
    df = df.drop_duplicates(subset=['Match_ID'])

# 2. Handle missing values in critical columns using a robust method (median)
if 'Player_Points' in df.columns:
    df['Player_Points'] = df['Player_Points'].fillna(df['Player_Points'].median())

if 'Match_Duration_Mins' in df.columns:
    df['Match_Duration_Mins'] = df['Match_Duration_Mins'].fillna(df['Match_Duration_Mins'].median())

# 3. Clean and normalize team and result names (stripping whitespace and standardizing case)
for col in ['Team', 'Result', 'Player']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Print cleaning summary
print("Data cleaning completed.")
print(f"Missing values remaining:\n{df.isnull().sum()}")

Data cleaning completed.
Missing values remaining:
Match_ID               0
Match_Date             0
Player                 0
Team                   0
Points                 5
Match_Duration         0
Result                 0
Player_Points          0
Match_Duration_Mins    0
dtype: int64


In [9]:
import pandas as pd

# 1. Convert Match_Date to datetime
df['Match_Date'] = pd.to_datetime(df['Match_Date'])

# 2. Extract Month (number: 1-12, or name: 'January', 'February', etc.)
df['Month'] = df['Match_Date'].dt.month
df['Month_Name'] = df['Match_Date'].dt.month_name()

# 3. Extract Weekday (number: Monday=0 to Sunday=6, or name: 'Monday', 'Tuesday', etc.)
df['Weekday'] = df['Match_Date'].dt.dayofweek
df['Weekday_Name'] = df['Match_Date'].dt.day_name()


In [10]:
# 1. Total matches played (counts total rows; use df['Match_ID'].nunique() if there are multiple rows per match)
total_matches = len(df)

# 2. Total points scored
total_points = df['Points'].sum()

# 3. Average match duration
avg_duration = df['Match_Duration'].mean()

print(f"Total Matches: {total_matches}")
print(f"Total Points: {total_points}")
print(f"Average Duration: {avg_duration:.2f}")


Total Matches: 100
Total Points: 12084.0
Average Duration: 119.31


In [11]:
# 1. Create a helper column for wins (1 if 'Win', else 0)
df['Is_Win'] = (df['Result'] == 'Win').astype(int)

# 2. Group by Team and calculate total points and wins
team_rankings = df.groupby('Team', as_index=False).agg(
    Total_Points=('Points', 'sum'),
    Matches_Won=('Is_Win', 'sum')
)

# 3. Sort by Total_Points (primary) and Matches_Won (tie-breaker)
team_rankings = team_rankings.sort_values(
    by=['Total_Points', 'Matches_Won'],
    ascending=[False, False]
).reset_index(drop=True)

# 4. Add a 1-based Rank column
team_rankings['Rank'] = team_rankings.index + 1

print(team_rankings)


               Team  Total_Points  Matches_Won  Rank
0  Bangalore Royals        3492.0           14     1
1   Kolkata Knights        3266.0           12     2
2     Mumbai Titans        2797.0            8     3
3    Delhi Strikers        2529.0           11     4


In [ ]:
top_10_players = (
    df.groupby(['Player', 'Team'], as_index=False)
    .agg(
        Total_Points=('Points', 'sum'),
        Matches_Played=('Match_ID', 'nunique')
    )
    .nlargest(10, 'Total_Points')
)


In [13]:
# Compare average points, match count, and spread across match results
if 'Result' in df.columns:
    result_comparison = (
        df.groupby('Result', as_index=False)
        .agg(
            Total_Matches=('Points', 'count'),
            Avg_Points=('Points', 'mean'),
            Median_Points=('Points', 'median'),
            Std_Dev=('Points', 'std')
        )
        .sort_values(by='Avg_Points', ascending=False)
        .reset_index(drop=True)
    )
    print(result_comparison.round(2))
else:
    print("Column 'Result' not found in DataFrame.")

  Result  Total_Matches  Avg_Points  Median_Points  Std_Dev
0   Draw              8      151.50          150.5    36.91
1    Win             42      126.52          129.5    46.32
2   Loss             45      123.51          131.0    44.11


In [16]:
# Group by Team to get total matches, wins, and win percentage
# (Note: df['Is_Win'] is already created in a previous cell)
team_win_pct = (
    df.groupby('Team', as_index=False)
    .agg(
        Matches_Played=('Result', 'count'),
        Matches_Won=('Is_Win', 'sum')
    )
)

team_win_pct['Win_Percentage'] = (
    (team_win_pct['Matches_Won'] / team_win_pct['Matches_Played']) * 100
).round(2)

# Sort from highest to lowest win percentage
team_win_pct = team_win_pct.sort_values(
    by=['Win_Percentage', 'Matches_Won'],
    ascending=[False, False]
).reset_index(drop=True)

print(team_win_pct)

               Team  Matches_Played  Matches_Won  Win_Percentage
0    Delhi Strikers              22           11           50.00
1   Kolkata Knights              26           12           46.15
2  Bangalore Royals              31           14           45.16
3     Mumbai Titans              21            8           38.10


In [15]:
# 1. Ensure Match_Duration is numeric or timedelta (if stored as strings like "01:45:00")
# df['Match_Duration'] = pd.to_timedelta(df['Match_Duration'])

# 2. Find exact min and max durations
max_duration = df['Match_Duration'].max()
min_duration = df['Match_Duration'].min()

# 3. Filter rows matching the longest and shortest durations (handles ties automatically)
longest_matches = df[df['Match_Duration'] == max_duration]
shortest_matches = df[df['Match_Duration'] == min_duration]

print("--- Longest Match(es) ---")
print(longest_matches)

print("\n--- Shortest Match(es) ---")
print(shortest_matches)


--- Longest Match(es) ---
   Match_ID Match_Date        Player             Team  Points  Match_Duration  \
70   M_1070 2023-01-21  Vihaan Patel  Kolkata Knights   148.0             179   

   Result  Player_Points  Match_Duration_Mins  Month Month_Name  Weekday  \
70   Draw          148.0                  179      1    January        5   

   Weekday_Name  Is_Win  
70     Saturday       0  

--- Shortest Match(es) ---
   Match_ID Match_Date        Player              Team  Points  \
85   M_1085 2023-01-18  Aarav Sharma    Delhi Strikers   116.0   
87   M_1087 2023-01-04   Kabir Singh  Bangalore Royals     NaN   
97   M_1097 2023-01-26   Kabir Singh   Kolkata Knights    88.0   

    Match_Duration Result  Player_Points  Match_Duration_Mins  Month  \
85              60    Win          116.0                   60      1   
87              60   Draw          135.0                   60      1   
97              60    Win           88.0                   60      1   

   Month_Name  Weekday W

In [17]:
# 1. Tuple for fixed, immutable match-result labels
MATCH_RESULTS = ('Win', 'Loss', 'Draw')

# Optional: Validate that the dataset only contains allowed result labels
invalid_results = set(df['Result'].dropna()) - set(MATCH_RESULTS)
if invalid_results:
    print(f"Warning: Unexpected result labels found: {invalid_results}")

# 2. Set to identify all unique participating teams (single 'Team' column)
participating_teams = set(df['Team'].dropna())

print(f"Allowed Results ({type(MATCH_RESULTS).__name__}): {MATCH_RESULTS}")
print(f"Total Unique Teams: {len(participating_teams)}")
print(f"Participating Teams ({type(participating_teams).__name__}): {sorted(participating_teams)}")


Allowed Results (tuple): ('Win', 'Loss', 'Draw')
Total Unique Teams: 4
Participating Teams (set): ['Bangalore Royals', 'Delhi Strikers', 'Kolkata Knights', 'Mumbai Titans']


In [18]:
# Tuple of fixed match-result labels from earlier
MATCH_RESULTS = ('Win', 'Loss', 'Draw')

# 1. Dictionary storing team-wise wins (ensuring teams with 0 wins are included as 0)
participating_teams = set(df['Team'].dropna())
win_counts = df[df['Result'] == MATCH_RESULTS[0]]['Team'].value_counts()
team_wins = {team: int(win_counts.get(team, 0)) for team in participating_teams}

# Dictionary storing total matches played per team
team_matches = df['Team'].value_counts().to_dict()

# 2. Function to calculate a team's win percentage
def get_win_percentage(team_name, wins_dict, matches_dict):
    """Calculate win percentage for a given team."""
    total_matches = matches_dict.get(team_name, 0)
    if total_matches == 0:
        return 0.0
    wins = wins_dict.get(team_name, 0)
    return round((wins / total_matches) * 100, 2)

# Example Usage
print("Team Wins Dictionary:", team_wins)

for team in sorted(participating_teams):
    win_pct = get_win_percentage(team, team_wins, team_matches)
    print(f"{team}: {team_wins[team]} wins / {team_matches[team]} matches ({win_pct}%)")


Team Wins Dictionary: {'Mumbai Titans': 8, 'Kolkata Knights': 12, 'Delhi Strikers': 11, 'Bangalore Royals': 14}
Bangalore Royals: 14 wins / 31 matches (45.16%)
Delhi Strikers: 11 wins / 22 matches (50.0%)
Kolkata Knights: 12 wins / 26 matches (46.15%)
Mumbai Titans: 8 wins / 21 matches (38.1%)


In [19]:
import numpy as np

# 1. Convert the Points column to a 1D NumPy array (dropping any missing values)
points_array = df['Points'].dropna().to_numpy()

# 2. Calculate mean, median, and standard deviation
mean_points = np.mean(points_array)
median_points = np.median(points_array)
std_points = np.std(points_array)  # Population std (ddof=0) by default

print(f"Mean Points: {mean_points:.2f}")
print(f"Median Points: {median_points:.2f}")
print(f"Standard Deviation: {std_points:.2f}")


Mean Points: 127.20
Median Points: 135.00
Standard Deviation: 44.54


In [20]:
import numpy as np
import pandas as pd

# 1. Calculate Q1, Q3, IQR, and the upper outlier fence using NumPy
points = df['Points'].dropna().to_numpy()
q1 = np.percentile(points, 25)
q3 = np.percentile(points, 75)
iqr = q3 - q1

# Standard Tukey upper fence for unusually high outliers
upper_fence = q3 + (1.5 * iqr)

print(f"Q1 (25th percentile): {q1:.2f}")
print(f"Q3 (75th percentile): {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Unusually High Score Threshold (> Q3 + 1.5*IQR): {upper_fence:.2f}")

# 2. Identify unusually high player scores
high_outliers = df[df['Points'] > upper_fence].sort_values(by='Points', ascending=False)
print("\n--- Unusually High Scores ---")
print(high_outliers[['Player', 'Points']])

# 3. Create player-performance groups using documented IQR/quartile thresholds
# Thresholds:
#   - Below Average : Points < Q1
#   - Average       : Q1 <= Points <= Q3
#   - Above Average : Q3 < Points <= Upper Fence (Q3 + 1.5 * IQR)
#   - Elite Outlier : Points > Upper Fence (Q3 + 1.5 * IQR)

bins = [-np.inf, q1, q3, upper_fence, np.inf]
labels = ['Below Average', 'Average', 'Above Average', 'Elite Outlier']

df['Performance_Group'] = pd.cut(
    df['Points'],
    bins=bins,
    labels=labels,
    right=True  # Intervals are (a, b]
)

print("\n--- Performance Group Distribution ---")
print(df['Performance_Group'].value_counts())


Q1 (25th percentile): 86.00
Q3 (75th percentile): 165.00
IQR: 79.00
Unusually High Score Threshold (> Q3 + 1.5*IQR): 283.50

--- Unusually High Scores ---
Empty DataFrame
Columns: [Player, Points]
Index: []

--- Performance Group Distribution ---
Performance_Group
Average          47
Below Average    25
Above Average    23
Elite Outlier     0
Name: count, dtype: int64


In [21]:
class Player:
    """Represents a tournament player with basic performance attributes."""

    def __init__(self, name: str, team: str, points: float):
        self.name = name
        self.team = team
        self.points = points

    def add_points(self, new_points: float) -> None:
        """Add points scored in a new match."""
        self.points += new_points

    def __repr__(self) -> str:
        return f"Player(name='{self.name}', team='{self.team}', points={self.points})"


# Example: Creating Player objects from a pandas DataFrame (e.g., top_10_players)
# players = [
#     Player(row['Player'], row['Team'], row['Total_Points'])
#     for _, row in player_summary.iterrows()
# ]

# Standalone usage example
p1 = Player("Aarav Sharma", "Delhi Strikers", 145)
p1.add_points(22)
print(p1)  # Output: Player(name='Aarav Sharma', team='Delhi Strikers', points=167)


Player(name='Aarav Sharma', team='Delhi Strikers', points=167)
